# Exercise 10 — UART hardware test bench
PYNQ-Z2 · Internal UART loopback · 50 MHz · 115200 baud · 8N1

## 1. Prerequisites
- Program the matching `design_1_wrapper` bitstream with Vivado before accessing MMIO.
- Run on the PYNQ board with Linux and Jupyter running. The notebook must be the sole controller of the bridge.
- FCLK0 must be dedicated to this design or safe to change to 50 MHz.
- Install the test dependencies once using the optional command below. Restart the kernel if needed, then execute the notebook in order.

A kernel restart does not reset the FPGA. Tests can be repeated without resetting the bridge. If a transaction times out or fails, stop and investigate before retrying. A Python timeout cannot interrupt a blocked AXI read.

In [1]:
# Optional: uncomment for the first installation only.
%pip install "pytest>=7,<9" "ipytest==0.14.2"

You should consider upgrading via the '/usr/local/share/pynq-venv/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 2. Imports and configuration

In [2]:
import math
import random
import time

import pytest
import ipytest
from pynq import MMIO, Clocks

ipytest.autoconfig(raise_on_error=True, addopts=())

EXPECTED_CLK_MHZ = 50.0
BAUD_RATE = 115_200
COMMAND_BASE_ADDR = 0x41200000
STATUS_BASE_ADDR = 0x41210000
GPIO_RANGE = 0x10000
GPIO_DATA_OFFSET = 0x00

DATA_MASK = 0xFF
REQUEST_MASK = ACK_MASK = 1 << 8
ACTIVE_MASK = 1 << 9
TX_BUSY_MASK = 1 << 10
RX_BUSY_MASK = 1 << 11
TX_DONE_MASK = 1 << 12
RX_DONE_MASK = 1 << 13
FRAMING_ERROR_MASK = 1 << 14
BUSY_MASK = ACTIVE_MASK | TX_BUSY_MASK | RX_BUSY_MASK
DONE_MASK = TX_DONE_MASK | RX_DONE_MASK
RESERVED_MASK = 0xFFFF8000
STRESS_COUNT = 10_000
RANDOM_SEED = 42

## 3. Hardware initialization
The bitstream is loaded externally; no overlay is downloaded here. A zero status is expected only immediately after hardware reset, not after every notebook execution.

In [3]:
# Configure FCLK0 before accessing AXI GPIO.
print(f"Previous FCLK0: {Clocks.fclk0_mhz} MHz")
if not math.isclose(Clocks.fclk0_mhz, EXPECTED_CLK_MHZ, rel_tol=0.001):
    Clocks.fclk0_mhz = EXPECTED_CLK_MHZ

assert math.isclose(Clocks.fclk0_mhz, EXPECTED_CLK_MHZ, rel_tol=0.001)
print(f"Current FCLK0: {Clocks.fclk0_mhz} MHz")

command_gpio = MMIO(COMMAND_BASE_ADDR, GPIO_RANGE)
status_gpio = MMIO(STATUS_BASE_ADDR, GPIO_RANGE)

Previous FCLK0: 50.0 MHz
Current FCLK0: 50.0 MHz


## 4. UART driver
The command contains TX data (bits 7:0) and the request toggle (bit 8). The status contains RX data, ACK, ACTIVE, TX_BUSY, RX_BUSY, TX_DONE, RX_DONE and FRAMING_ERROR in bits 0–14. Bits 31:15 are reserved.

Each call reads the current ACK and inverts it, then waits for matching ACK, idle state and both completion flags. The driver returns the received byte; tests independently compare it to the transmitted byte.

In [4]:
def read_status() -> int:
    """Read the current hardware status, not a cached variable."""
    return int(status_gpio.read(GPIO_DATA_OFFSET))


def uart_transfer(data: int, timeout: float = 1.0) -> int:
    """Send one byte and wait for the internal loopback result."""
    if type(data) is not int or not 0 <= data <= DATA_MASK:
        raise ValueError("Data must be an 8-bit integer")
    if not math.isfinite(timeout) or timeout <= 0:
        raise ValueError("Timeout must be finite and positive")

    status = read_status()
    command = int(command_gpio.read(GPIO_DATA_OFFSET))
    if status & (BUSY_MASK | FRAMING_ERROR_MASK):
        raise RuntimeError(f"Bridge not ready: 0x{status:08X}")
    if (command ^ status) & ACK_MASK:
        raise RuntimeError("Previous request is still pending")

    request = (status & ACK_MASK) ^ REQUEST_MASK
    command_gpio.write(GPIO_DATA_OFFSET, request | data)
    deadline = time.monotonic() + timeout

    while True:
        status = read_status()
        if (status & ACK_MASK) == request and not (status & BUSY_MASK):
            if status & FRAMING_ERROR_MASK:
                raise RuntimeError(f"UART framing error: 0x{status:08X}")
            if (status & DONE_MASK) == DONE_MASK:
                return status & DATA_MASK
        if time.monotonic() >= deadline:
            raise TimeoutError(f"UART timeout: status=0x{status:08X}")

## 5. Hardware configuration tests
Verify the clock, idle state, reserved bits and command/ACK agreement. Sticky completion flags may remain set after earlier successful transfers.

In [5]:
%%ipytest -v -x

def test_clock_frequency():
    assert math.isclose(Clocks.fclk0_mhz, EXPECTED_CLK_MHZ, rel_tol=0.001)


def test_bridge_ready():
    status = read_status()
    command = int(command_gpio.read(GPIO_DATA_OFFSET))
    assert status & (BUSY_MASK | FRAMING_ERROR_MASK | RESERVED_MASK) == 0
    assert (command & REQUEST_MASK) == (status & ACK_MASK)
    print(f"Current status: 0x{status:08X}")

======================================= test session starts ========================================
platform linux -- Python 3.10.4, pytest-8.4.2, pluggy-1.7.0 -- /usr/local/share/pynq-venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/xilinx/jupyter_notebooks
plugins: anyio-3.6.1, dash-2.6.0, sourceorder-0.5.1
collecting ... collected 2 items

t_ad4380749b204db0b2b2a4356d61f3b9.py::test_clock_frequency PASSED                           [ 50%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_bridge_ready PASSED                              [100%]

======================================== 2 passed in 0.24s =========================================


## 6. Handshake and representative patterns
Check two consecutive transfers, including exact final status and both toggle directions, regardless of the initial ACK. Then test five representative byte patterns.

In [6]:
%%ipytest -v -x

def test_toggle_handshake():
    initial_ack = read_status() & ACK_MASK
    for data in (0xA6, 0x55):
        expected_ack = initial_ack ^ ACK_MASK
        assert uart_transfer(data) == data
        assert read_status() == DONE_MASK | expected_ack | data
        initial_ack = expected_ack


@pytest.mark.parametrize("data", [0x00, 0xFF, 0x55, 0xAA, 0xA6],
                         ids=lambda data: f"0x{data:02X}")
def test_uart_pattern(data):
    assert uart_transfer(data) == data

======================================= test session starts ========================================
platform linux -- Python 3.10.4, pytest-8.4.2, pluggy-1.7.0 -- /usr/local/share/pynq-venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/xilinx/jupyter_notebooks
plugins: anyio-3.6.1, dash-2.6.0, sourceorder-0.5.1
collecting ... collected 6 items

t_ad4380749b204db0b2b2a4356d61f3b9.py::test_toggle_handshake PASSED                          [ 16%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_pattern[0x00] PASSED                        [ 33%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_pattern[0xFF] PASSED                        [ 50%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_pattern[0x55] PASSED                        [ 66%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_pattern[0xAA] PASSED                        [ 83%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_pattern[0xA6] PASSED                        [100%]

======================================== 6 pas

## 7. Exhaustive byte test
Generate one independently reported pytest case for every possible 8-bit value. Stop on the first failure.

In [7]:
%%ipytest -v -x

@pytest.mark.parametrize("data", range(256), ids=lambda data: f"0x{data:02X}")
def test_uart_exhaustive(data):
    assert uart_transfer(data) == data

======================================= test session starts ========================================
platform linux -- Python 3.10.4, pytest-8.4.2, pluggy-1.7.0 -- /usr/local/share/pynq-venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/xilinx/jupyter_notebooks
plugins: anyio-3.6.1, dash-2.6.0, sourceorder-0.5.1
collecting ... collected 256 items

t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_exhaustive[0x00] PASSED                     [  0%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_exhaustive[0x01] PASSED                     [  0%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_exhaustive[0x02] PASSED                     [  1%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_exhaustive[0x03] PASSED                     [  1%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_exhaustive[0x04] PASSED                     [  1%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_exhaustive[0x05] PASSED                     [  2%]
t_ad4380749b204db0b2b2a4356d61f3b9.py::test_u

## 8. Stress test
Run 10,000 consecutive pseudo-random transfers as one test. A fixed seed makes failures reproducible. Timing excludes pytest per-case reporting overhead.

In [8]:
%%ipytest -v -x -s

def test_uart_stress():
    rng = random.Random(RANDOM_SEED)
    start = time.perf_counter()
    for index in range(STRESS_COUNT):
        data = rng.randrange(256)
        received = uart_transfer(data)
        assert received == data, f"Transfer {index}: TX=0x{data:02X}, RX=0x{received:02X}"

    elapsed = time.perf_counter() - start
    print(f"\nPassed: {STRESS_COUNT}/{STRESS_COUNT}")
    print(f"Execution time: {elapsed:.4f} s")
    print(f"Transfers per second: {STRESS_COUNT / elapsed:.2f}")

======================================= test session starts ========================================
platform linux -- Python 3.10.4, pytest-8.4.2, pluggy-1.7.0 -- /usr/local/share/pynq-venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/xilinx/jupyter_notebooks
plugins: anyio-3.6.1, dash-2.6.0, sourceorder-0.5.1
collecting ... collected 1 item

t_ad4380749b204db0b2b2a4356d61f3b9.py::test_uart_stress 
Passed: 10000/10000
Execution time: 1.9564 s
Transfers per second: 5111.46
PASSED

======================================== 1 passed in 2.12s =========================================


## 9. End-to-end performance
Measure the Python/MMIO/FPGA round trip without pytest collecting a separate test for each byte. This is not a direct measurement of serial baud rate.

In [9]:
# Measure and validate all byte values in a single timed loop.
start = time.perf_counter()
for data in range(256):
    assert uart_transfer(data) == data
elapsed = time.perf_counter() - start

print(f"Transfers: 256")
print(f"Execution time: {elapsed:.4f} s")
print(f"Average latency: {elapsed / 256 * 1e6:.2f} us")
print(f"Throughput: {256 / elapsed:.2f} bytes/s")
print(f"Ideal 8N1 frame duration: {10 / BAUD_RATE * 1e6:.2f} us")

Transfers: 256
Execution time: 0.0520 s
Average latency: 203.09 us
Throughput: 4923.81 bytes/s
Ideal 8N1 frame duration: 86.81 us


## 10. Validation scope
This notebook verifies the internal UART loopback and software/hardware handshake on the programmed FPGA. It does not validate external UART pins, independent transmitter/receiver clock mismatch, framing-error injection, or physical baud rate using an instrument. RTL simulation remains separate.

Each `%%ipytest` cell reports its own test group and clears previously defined test functions. The performance cell is a separate measurement, not a pytest group.